In [1]:
import json
import os
import re
import boto3
import pandas as pd
import logging

In [6]:
pd.set_option('display.max_rows', None)

In [2]:
with open("/home/rakshith/Data/IPL-2026/data/09_GTvsRR_meta.json", "r") as f:
    data = f.read()
metadata = json.loads(data)


df = pd.read_csv("/home/rakshith/Data/IPL-2026/data/09_GTvsRR_data.csv")

In [3]:
def get_toss_info(metadata):
    print("\nGetting Toss Info:\n")
    team1 = metadata["home_team"]
    team2 = metadata["away_team"]
    if metadata["toss_winner"] == team1:
        if metadata["toss_decision"] == "bat":
            batting_team = team1
            bowling_team = team2
        else:
            batting_team = team2
            bowling_team = team1
    else:
        if metadata["toss_decision"] == "bat":
            batting_team = team2
            bowling_team = team1
        else:
            batting_team = team1
            bowling_team = team2

    print(f"Batting: {batting_team}\nBowling: {bowling_team}\n")
    return batting_team, bowling_team

def get_innings(last_ball, new_ball):
    print("\nGetting Innings:\n")
    if last_ball == None:
        innings = 1
        score = 0
        wickets = 0
        target = 0
    else:
        if last_ball["innings"] == 1 and (int(last_ball["over"]) - int(new_ball["over"]) > 5):
            innings = 2
            score = 0
            wickets = 0
            target = last_ball["current_score"]
        elif last_ball["innings"] == 2 and (int(last_ball["over"]) - int(new_ball["over"]) > 5):
            innings = 3
            score = 0
            wickets = 0
            target = 0
        elif last_ball["innings"] == 3 and (int(last_ball["ball"]) - int(new_ball["ball"]) > 3):
            innings = 4
            score = 0
            wickets = 0
            target = last_ball["current_score"]
        else:
            innings = last_ball["innings"]
            score = last_ball["current_score"]
            wickets = last_ball["current_wickets"]
            target = last_ball["target"]
            
    return innings, score, wickets, target

In [21]:
def extract_runs(df):
    run_dict = {
        'no run': 0,
        '1 run': 1,
        '2 runs': 2,
        '3 runs': 3,
        'FOUR': 4,
        'four': 4,
        'SIX': 6,
        'six': 6
    }
    extras_dict = {
        'wide': 1,
        'no ball': 1,
        'no-ball': 1,
        'byes': 0,
        'leg byes': 0,
        'leg-byes': 0
    }

    runs = 0
    extra_runs = 0
    extra = 0
    extra_type = "N/A"
    rebowl = 0
    wicket = 0
    wicket_method = "Not Out"
    out_batsman = "N/A"

    ball_event = df['ball_event']
    event_info = df['event_info']

    if ball_event in run_dict.keys():
        runs = run_dict[ball_event]

    if ball_event in extras_dict.keys():
        extra = 1
        extra_runs = extras_dict[ball_event]
        rebowl = extras_dict[ball_event]
        extra_type = ball_event
        extra_event = event_info.split(';')[0]
        if extra_event in run_dict.keys():
            runs = run_dict[extra_event]

    elif ball_event == '5 wides':
        runs = 4
        extra_runs = 1
        extra = 1
        extra_type = 'wide'
        rebowl = 1

    elif ball_event.startswith('out'):
        wicket = 1
        if 'Run Out!' in ball_event:
            wicket_method = 'Run Out'
            out_batsman = ball_event.split('out ')[-1].split(' Run Out')[0]
        else:
            wicket_method = ball_event.split(' ')[1]
            out_batsman = df['batsman']

    data = {}
    data['match'] = df['match']
    data['data'] = df['date']
    data['time'] = df['time']
    data['over'] = df['over']
    data['ball'] = df['ball']
    data['bowler'] = df['bowler']
    data['batsman'] = df['batsman']
    data['runs'] = runs
    data['extra_runs'] = extra_runs
    data['extra'] = extra
    data['extra_type'] = extra_type
    data['rebowl'] = rebowl
    data['wicket'] = wicket
    data['wicket_method'] = wicket_method
    data['out_batsman'] = out_batsman
    data['total_runs'] = runs + extra_runs

    return data

In [22]:
extract_data = []

In [23]:
# batting_team, bowling_team = get_toss_info(metadata)

for index, row in df.iterrows():
    print(f"Extracting for: {row['over']}.{row['ball']}")
    extract_data.append(extract_runs(row))

Extracting for: 0.1
Extracting for: 0.1
Extracting for: 0.2
Extracting for: 0.3
Extracting for: 0.2
Extracting for: 0.3
Extracting for: 0.3
Extracting for: 0.4
Extracting for: 0.3
Extracting for: 0.4
Extracting for: 0.4
Extracting for: 0.5
Extracting for: 0.4
Extracting for: 0.5
Extracting for: 0.5
Extracting for: 0.6
Extracting for: 0.5
Extracting for: 0.6
Extracting for: 1.1
Extracting for: 1.2
Extracting for: 1.1
Extracting for: 1.2
Extracting for: 1.3
Extracting for: 1.4
Extracting for: 1.3
Extracting for: 1.4
Extracting for: 1.4
Extracting for: 1.4
Extracting for: 1.4
Extracting for: 1.4
Extracting for: 1.5
Extracting for: 1.6
Extracting for: 1.5
Extracting for: 1.6
Extracting for: 2.1
Extracting for: 2.2
Extracting for: 1.6
Extracting for: 2.1
Extracting for: 2.2
Extracting for: 2.3
Extracting for: 2.2
Extracting for: 2.3
Extracting for: 2.3
Extracting for: 2.3
Extracting for: 2.3
Extracting for: 2.3
Extracting for: 2.3
Extracting for: 2.4
Extracting for: 2.3
Extracting for: 2.4


In [24]:
extract_df = pd.DataFrame(extract_data)

In [25]:
extract_df.head()

,match,data,time,over,ball,bowler,batsman,runs,extra_runs,extra,extra_type,rebowl,wicket,wicket_method,out_batsman,total_runs
0,09_GTvsRR,April 04,19:00:01,0,1,Mohammed Siraj,Yashasvi Jaiswal,1,0,0,N/A,0,0,Not Out,N/A,1
1,09_GTvsRR,April 04,19:00:01,0,1,Mohammed Siraj,Yashasvi Jaiswal,1,0,0,N/A,0,0,Not Out,N/A,1
2,09_GTvsRR,April 04,19:00:01,0,2,Mohammed Siraj,Vaibhav Sooryavanshi,4,0,0,N/A,0,0,Not Out,N/A,4
3,09_GTvsRR,April 04,19:00:01,0,3,Mohammed Siraj,Vaibhav Sooryavanshi,0,0,0,N/A,0,0,Not Out,N/A,0
4,09_GTvsRR,April 04,19:00:01,0,2,Mohammed Siraj,Vaibhav Sooryavanshi,4,0,0,N/A,0,0,Not Out,N/A,4


In [26]:
extract_df.shape

(622, 16)

In [27]:
df2 = extract_df.drop_duplicates().reset_index(drop=True)

In [28]:
df2.head(500)

,match,data,time,over,ball,bowler,batsman,runs,extra_runs,extra,extra_type,rebowl,wicket,wicket_method,out_batsman,total_runs
0,09_GTvsRR,April 04,19:00:01,0,1,Mohammed Siraj,Yashasvi Jaiswal,1,0,0,N/A,0,0,Not Out,N/A,1
1,09_GTvsRR,April 04,19:00:01,0,2,Mohammed Siraj,Vaibhav Sooryavanshi,4,0,0,N/A,0,0,Not Out,N/A,4
2,09_GTvsRR,April 04,19:00:01,0,3,Mohammed Siraj,Vaibhav Sooryavanshi,0,0,0,N/A,0,0,Not Out,N/A,0
3,09_GTvsRR,April 04,19:00:01,0,4,Mohammed Siraj,Vaibhav Sooryavanshi,4,0,0,N/A,0,0,Not Out,N/A,4
4,09_GTvsRR,April 04,19:00:01,0,5,Mohammed Siraj,Vaibhav Sooryavanshi,1,0,0,N/A,0,0,Not Out,N/A,1
5,09_GTvsRR,April 04,19:00:01,0,6,Mohammed Siraj,Yashasvi Jaiswal,0,0,0,N/A,0,0,Not Out,N/A,0
6,09_GTvsRR,April 04,19:00:01,1,1,Kagiso Rabada,Vaibhav Sooryavanshi,1,0,0,N/A,0,0,Not Out,N/A,1
7,09_GTvsRR,April 04,19:00:01,1,2,Kagiso Rabada,Yashasvi Jaiswal,1,0,0,N/A,0,0,Not Out,N/A,1
8,09_GTvsRR,April 04,19:00:01,1,3,Kagiso Rabada,Vaibhav Sooryavanshi,0,0,0,N/A,0,0,Not Out,N/A,0
9,09_GTvsRR,April 04,19:00:01,1,4,Kagiso Rabada,Vaibhav Sooryavanshi,0,1,1,wide,1,0,Not Out,N/A,1


In [52]:
final = []

for index, row in df2.iterrows():

    row_dict = row.to_dict()

    if final == []:
        row_dict['innings'] = 1
        row_dict['score'] = row_dict['total_runs']
        row_dict['fallen_wickets'] = row_dict['wicket']

    else:
        prev_ball_innings = int(final[-1]['innings'])
        if row_dict['over'] == 0 and row_dict['ball'] == 1:
            if final[-1]['rebowl'] == 0:
                row_dict['innings'] = prev_ball_innings + 1 
                row_dict['score'] = 0
                row_dict['fallen_wickets'] = 0  
            else: row_dict['innings'] = prev_ball_innings
        else: row_dict['innings'] = prev_ball_innings        

        if row_dict['innings'] == prev_ball_innings:
            row_dict['score'] = final[-1]['score'] + row_dict['total_runs']
            row_dict['fallen_wickets'] = final[-1]['fallen_wickets'] + row_dict['wicket']
        else:
            row_dict['score'] = 0
            row_dict['fallen_wickets'] = 0

    final.append(row_dict)


In [53]:
pd.DataFrame(final)

,match,data,time,over,ball,bowler,batsman,runs,extra_runs,extra,extra_type,rebowl,wicket,wicket_method,out_batsman,total_runs,innings,score,fallen_wickets
0,09_GTvsRR,April 04,19:00:01,0,1,Mohammed Siraj,Yashasvi Jaiswal,1,0,0,N/A,0,0,Not Out,N/A,1,1,1,0
1,09_GTvsRR,April 04,19:00:01,0,2,Mohammed Siraj,Vaibhav Sooryavanshi,4,0,0,N/A,0,0,Not Out,N/A,4,1,5,0
2,09_GTvsRR,April 04,19:00:01,0,3,Mohammed Siraj,Vaibhav Sooryavanshi,0,0,0,N/A,0,0,Not Out,N/A,0,1,5,0
3,09_GTvsRR,April 04,19:00:01,0,4,Mohammed Siraj,Vaibhav Sooryavanshi,4,0,0,N/A,0,0,Not Out,N/A,4,1,9,0
4,09_GTvsRR,April 04,19:00:01,0,5,Mohammed Siraj,Vaibhav Sooryavanshi,1,0,0,N/A,0,0,Not Out,N/A,1,1,10,0
5,09_GTvsRR,April 04,19:00:01,0,6,Mohammed Siraj,Yashasvi Jaiswal,0,0,0,N/A,0,0,Not Out,N/A,0,1,10,0
6,09_GTvsRR,April 04,19:00:01,1,1,Kagiso Rabada,Vaibhav Sooryavanshi,1,0,0,N/A,0,0,Not Out,N/A,1,1,11,0
7,09_GTvsRR,April 04,19:00:01,1,2,Kagiso Rabada,Yashasvi Jaiswal,1,0,0,N/A,0,0,Not Out,N/A,1,1,12,0
8,09_GTvsRR,April 04,19:00:01,1,3,Kagiso Rabada,Vaibhav Sooryavanshi,0,0,0,N/A,0,0,Not Out,N/A,0,1,12,0
9,09_GTvsRR,April 04,19:00:01,1,4,Kagiso Rabada,Vaibhav Sooryavanshi,0,1,1,wide,1,0,Not Out,N/A,1,1,13,0


In [7]:
df = pd.read_json("/home/rakshith/Data/IPL-2026/data/10_SHvsLSG/10_SHvsLSG_silver.json")
df.head(500)

,match,data,time,over,ball,bowler,batsman,runs,extra_runs,extra,extra_type,rebowl,wicket,wicket_method,out_batsman,total_runs,innings,score,fallen_wickets
0,10_SHvsLSG,April 05,15:30:02,0,1,Mohammed Shami,Travis Head,0,0,0,N/A,0,0,Not Out,N/A,0,1,0,0
1,10_SHvsLSG,April 05,15:30:02,0,2,Mohammed Shami,Travis Head,0,0,0,N/A,0,0,Not Out,N/A,0,1,0,0
2,10_SHvsLSG,April 05,15:30:02,0,3,Mohammed Shami,Travis Head,0,0,0,N/A,0,0,Not Out,N/A,0,1,0,0
3,10_SHvsLSG,April 05,15:30:02,0,3,Mohammed Shami,Travis Head,0,0,0,N/A,0,0,Not Out,N/A,0,1,0,0
4,10_SHvsLSG,April 05,15:30:02,0,4,Mohammed Shami,Travis Head,1,0,0,N/A,0,0,Not Out,N/A,1,1,1,0
5,10_SHvsLSG,April 05,15:30:02,0,4,Mohammed Shami,Travis Head,1,0,0,N/A,0,0,Not Out,N/A,1,1,2,0
6,10_SHvsLSG,April 05,15:30:02,0,5,Mohammed Shami,Abhishek Sharma,0,0,0,N/A,0,0,Not Out,N/A,0,1,2,0
7,10_SHvsLSG,April 05,15:30:02,0,4,Mohammed Shami,Travis Head,1,0,0,N/A,0,0,Not Out,N/A,1,1,3,0
8,10_SHvsLSG,April 05,15:30:02,0,5,Mohammed Shami,Abhishek Sharma,0,0,0,N/A,0,0,Not Out,N/A,0,1,3,0
9,10_SHvsLSG,April 05,15:30:02,0,5,Mohammed Shami,Abhishek Sharma,0,0,0,N/A,0,0,Not Out,N/A,0,1,3,0
